# Customer Platform Choice Prediction

**Python · PyTorch · Multiclass classification**

**Samuel Kalisch · Alice Finotti · Esteban Cantu · Lucia Bonnin · Ludovico Panariello**

Predict customers' selection of one of three shopping platforms (A, B, or C) using customer attributes and each platform's delivery time and cost. The model uses a shared neural-network encoder to score alternatives on the same scale.

## 1. Data and exploratory analysis

The dataset contains 7,000 choice scenarios. `GROUP` and `MEMBER` describe the customer; `DT_A`/`COST_A`, `DT_B`/`COST_B`, and `DT_C`/`COST_C` describe the alternatives. `CHOICE` identifies the selected platform (0, 1, or 2).

In [ ]:
from pathlib import Path
import os
import random
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BATCH_SIZE = 100
MAX_EPOCHS = int(os.environ.get("CHOICE_EPOCHS", "100"))
PATIENCE = 15
DATA_URL = "https://raw.githubusercontent.com/ecpm0/colab-files/refs/heads/main/training_data.csv"
DATA_PATH = Path("training_data.csv")

sns.set_theme(style="whitegrid")

In [ ]:
try:
    df = pd.read_csv(DATA_PATH if DATA_PATH.exists() else DATA_URL)
except Exception as exc:
    raise RuntimeError("Could not load training data. Download training_data.csv from the data URL and place it beside the notebook.") from exc

required_columns = ["GROUP", "MEMBER", "DT_A", "COST_A", "DT_B", "COST_B", "DT_C", "COST_C", "CHOICE"]
if not set(required_columns).issubset(df.columns):
    raise ValueError("Missing required columns: " + ", ".join(sorted(set(required_columns) - set(df.columns))))
df = df[required_columns].dropna().copy()
if not set(df["CHOICE"].unique()).issubset({0, 1, 2}):
    raise ValueError("CHOICE must contain class labels 0, 1, and 2.")
print(f"Observations: {len(df):,}")
display(df.describe().round(2))
print("Platform choice distribution:")
display(df["CHOICE"].value_counts(normalize=True).sort_index().rename("proportion").round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df[["DT_A", "DT_B", "DT_C"]], ax=axes[0], showfliers=False)
axes[0].set(title="Delivery time by platform", ylabel="Delivery time")
sns.boxplot(data=np.log1p(df[["COST_A", "COST_B", "COST_C"]].clip(lower=0)), ax=axes[1], showfliers=False)
axes[1].set(title="Log(1 + cost) by platform", ylabel="Log-transformed cost")
plt.tight_layout()
plt.show()

## 2. Alternative-level representation and data splits

Each observation is represented as a **3 × 4 tensor**: three platform alternatives, each with customer `GROUP`, customer `MEMBER`, delivery time, and cost. The same feature scaler is fitted to all platform rows **using training data only**, preserving the shared encoder's permutation-equivariant structure.

An 80/20 development–test split is followed by a validation split within the development portion. The test data are held out until the final evaluation.

In [ ]:
customer = df[["GROUP", "MEMBER"]].to_numpy(dtype=np.float32)
platform = np.stack([df[[f"DT_{p}", f"COST_{p}"]].to_numpy(dtype=np.float32) for p in "ABC"], axis=1)
X = np.concatenate([np.repeat(customer[:, None, :], 3, axis=1), platform], axis=2)
y = df["CHOICE"].to_numpy(dtype=np.int64)
assert X.shape == (len(df), 3, 4)

idx = np.arange(len(y))
dev_idx, test_idx = train_test_split(idx, test_size=0.20, stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(dev_idx, test_size=0.20, stratify=y[dev_idx], random_state=SEED)

# A common transformation for all alternatives retains permutation equivariance.
shared_scaler = StandardScaler().fit(X[train_idx].reshape(-1, 4))
def scale_alternatives(values):
    return shared_scaler.transform(values.reshape(-1, 4)).reshape(values.shape).astype(np.float32)

X_train, X_val, X_test = (scale_alternatives(X[ix]) for ix in (train_idx, val_idx, test_idx))
y_train, y_val, y_test = (y[ix] for ix in (train_idx, val_idx, test_idx))

def make_loader(features, labels, shuffle=False):
    dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(labels))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)
print(f"Train: {len(y_train):,} | Validation: {len(y_val):,} | Test: {len(y_test):,}")

## 3. Multinomial logistic regression benchmark

A multinomial logistic regression on the flattened 12-feature representation provides an interpretable baseline. Standardization is fitted on the training partition only.

In [ ]:
benchmark = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, solver="lbfgs"))
benchmark.fit(X[train_idx].reshape(-1, 12), y_train)
base_train = accuracy_score(y_train, benchmark.predict(X[train_idx].reshape(-1, 12)))
base_val = accuracy_score(y_val, benchmark.predict(X[val_idx].reshape(-1, 12)))
print(f"Logistic regression | train accuracy: {base_train:.3f} | validation accuracy: {base_val:.3f}")

## 4. Shared-encoder neural network

The **same MLP** is applied independently to each platform: 4 → 32 → 16 → 1, with ReLU activations. The three resulting scores are concatenated and passed to cross-entropy loss. Since all platform rows share one encoder, swapping platform rows swaps the corresponding output scores (**permutation equivariance**).

In [ ]:
class ChoiceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(4, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        # x: (batch, 3 alternatives, 4 features)
        return self.encoder(x).squeeze(-1)

print(f"Trainable parameters: {sum(p.numel() for p in ChoiceModel().parameters()):,}")

## 5. Hyperparameter search and early stopping

Five Adam configurations compare learning rate, momentum terms, and weight decay. Models are trained with cross-entropy loss and **early stopping based on validation loss** (patience 15). Each configuration starts with the same seed, and its best checkpoint is retained.

In [ ]:
HYPERPARAMETERS = [
    {"lr": 0.0020, "betas": (0.90, 0.980), "weight_decay": 2e-5},
    {"lr": 0.0018, "betas": (0.90, 0.980), "weight_decay": 5e-5},
    {"lr": 0.0022, "betas": (0.90, 0.980), "weight_decay": 5e-5},
    {"lr": 0.0020, "betas": (0.92, 0.985), "weight_decay": 5e-5},
    {"lr": 0.0020, "betas": (0.90, 0.980), "weight_decay": 0.0},
]

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            total_loss += loss_fn(logits, yb).item() * len(yb)
            correct += int((logits.argmax(dim=1) == yb).sum())
            n += len(yb)
    return total_loss / n, correct / n

def train_one(config):
    torch.manual_seed(SEED)
    model = ChoiceModel()
    optimizer = torch.optim.Adam(model.parameters(), **config)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "validation_loss": [], "validation_accuracy": []}
    best_loss = float("inf")
    best_state = None
    stagnant_epochs = 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        total, n = 0.0, 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += float(loss.item()) * len(yb)
            n += len(yb)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history["train_loss"].append(total / n)
        history["validation_loss"].append(val_loss)
        history["validation_accuracy"].append(val_acc)
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            stagnant_epochs = 0
        else:
            stagnant_epochs += 1
            if stagnant_epochs >= PATIENCE:
                break
    model.load_state_dict(best_state)
    final_loss, final_acc = evaluate(model, val_loader, criterion)
    return model, history, final_loss, final_acc

search_results = []
models = []
for number, hp in enumerate(HYPERPARAMETERS, start=1):
    candidate, history, val_loss, val_acc = train_one(hp)
    search_results.append({"config": number, **hp, "validation_loss": val_loss, "validation_accuracy": val_acc, "epochs": len(history["train_loss"])})
    models.append((candidate, history))
    print(f"Config {number}: validation accuracy={val_acc:.3f}, loss={val_loss:.3f}, epochs={len(history['train_loss'])}")
results = pd.DataFrame(search_results)
display(results)
best_index = int(results["validation_loss"].argmin())
best_model, best_history = models[best_index]
print(f"Selected configuration: {best_index + 1} | {HYPERPARAMETERS[best_index]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(best_history["train_loss"], label="Training")
axes[0].plot(best_history["validation_loss"], label="Validation")
axes[0].set(title="Cross-entropy loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[1].plot(best_history["validation_accuracy"])
axes[1].set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
plt.tight_layout()
plt.show()

## 6. Held-out test evaluation

The test split is used only after model selection. Results are compared with the logistic baseline on the **same test observations**.

In [ ]:
benchmark_test = accuracy_score(y_test, benchmark.predict(X[test_idx].reshape(-1, 12)))
best_model.eval()
with torch.no_grad():
    test_logits = best_model(torch.from_numpy(X_test))
    test_pred = test_logits.argmax(dim=1).numpy()
    test_prob = torch.softmax(test_logits, dim=1).numpy()
network_test = accuracy_score(y_test, test_pred)
print(f"Logistic regression test accuracy: {benchmark_test:.3f}")
print(f"Shared-encoder MLP test accuracy: {network_test:.3f}")
print(classification_report(y_test, test_pred, target_names=["Platform A", "Platform B", "Platform C"]))
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, test_pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set(xlabel="Predicted platform", ylabel="Actual platform", title="Neural network: held-out confusion matrix")
plt.tight_layout()
plt.show()

## 7. Permutation-equivariance check

When two platform rows are exchanged, the corresponding predicted probabilities should exchange positions, while the third probability stays unchanged. This follows from sharing encoder weights across alternatives.

In [ ]:
x_example = torch.from_numpy(X_test[:12])
x_swapped = x_example[:, [1, 0, 2], :]
best_model.eval()
with torch.no_grad():
    original = torch.softmax(best_model(x_example), dim=1)
    swapped = torch.softmax(best_model(x_swapped), dim=1)
assert torch.allclose(swapped, original[:, [1, 0, 2]], atol=1e-6)
print("Permutation-equivariance check passed.")
print("Example probabilities:", original[0].numpy().round(3))